# Comparative Analysis of Numerical Methods for Black-Scholes PDE

This notebook performs a comparative analysis of the implemented numerical methods (Analytical Black-Scholes, Binomial Tree, Monte Carlo, and Finite Difference) for option pricing. It will load data from the SQLite database, compare the results, and visualize key findings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import sys

# Add the project root to sys.path for importing modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from src.data import database
from src.options.base import Option
from src.analytical.black_scholes import BlackScholes

# --- Configuration ---
DATABASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'option_pricing_data.db'))
print(f"Database path: {DATABASE_PATH}")

# --- Load Data ---
conn = database.connect_db(DATABASE_PATH)

options_df = pd.read_sql_query("SELECT * FROM options", conn)
results_df = pd.read_sql_query("SELECT * FROM results", conn)
market_data_df = pd.read_sql_query("SELECT * FROM market_data", conn)

conn.close()

print("Data loaded successfully:")
print(f"Options: {len(options_df)} records")
print(f"Results: {len(results_df)} records")
print(f"Market Data: {len(market_data_df)} records")

# Merge options and results for easier analysis
merged_df = pd.merge(results_df, options_df, left_on='option_id', right_on='id', suffixes=('_result', '_option'))
merged_df.rename(columns={'id_option': 'option_params_id'}, inplace=True)

merged_df.head()

## 1. Price Comparison: Analytical vs. Numerical Methods

In [ ]:
# Filter for a specific option (e.g., the first call option in the database)
example_option_id = options_df[options_df['option_type'] == 'call'].iloc[0]['id']
example_option_data = merged_df[merged_df['option_id'] == example_option_id]

plt.figure(figsize=(10, 6))
sns.barplot(x='method', y='price', data=example_option_data)
plt.title(f'Price Comparison for Option ID {example_option_id}')
plt.ylabel('Option Price')
plt.xlabel('Pricing Method')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')
plt.tight_layout()
plt.show()

## 2. Error Analysis (vs. Analytical Black-Scholes)

In [ ]:
# Get analytical price for comparison
analytical_price_row = example_option_data[example_option_data['method'] == 'BlackScholes']
if not analytical_price_row.empty:
    analytical_price = analytical_price_row['price'].iloc[0]
    
    # Calculate absolute error
    example_option_data['absolute_error'] = abs(example_option_data['price'] - analytical_price)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='method', y='absolute_error', data=example_option_data[example_option_data['method'] != 'BlackScholes'])
    plt.title(f'Absolute Error vs. Black-Scholes for Option ID {example_option_id}')
    plt.ylabel('Absolute Error')
    plt.xlabel('Numerical Method')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()
else:
    print("Analytical Black-Scholes price not found for error analysis.")

## 3. Computational Time Comparison

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='method', y='computation_time', data=example_option_data)
plt.title(f'Computational Time Comparison for Option ID {example_option_id}')
plt.ylabel('Time (seconds)')
plt.xlabel('Pricing Method')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')
plt.yscale('log') # Use log scale for better visualization if times vary widely
plt.tight_layout()
plt.show()

## 4. Implied Volatility Comparison (Market Data vs. Model)

In [ ]:
# This section requires more sophisticated analysis, potentially iterating through market data
# and calculating implied volatility from model prices, or comparing model IV to market IV.
# For now, we'll just show the raw implied volatility from market data.

if not market_data_df.empty:
    # Filter for a specific ticker and expiration for clarity
    spy_market_data = market_data_df[market_data_df['ticker'] == 'SPY']
    if not spy_market_data.empty:
        latest_expiration = spy_market_data['expiration_date'].max()
        spy_market_data_latest_exp = spy_market_data[spy_market_data['expiration_date'] == latest_expiration]
        
        plt.figure(figsize=(12, 7))
        sns.scatterplot(x='strike_price', y='implied_volatility', hue='option_type', data=spy_market_data_latest_exp)
        plt.title(f'Implied Volatility Smile/Skew for SPY (Exp: {latest_expiration})')
        plt.xlabel('Strike Price')
        plt.ylabel('Implied Volatility')
        plt.grid(True, linestyle='--')
        plt.tight_layout()
        plt.show()
    else:
        print("No SPY market data found for implied volatility analysis.")
else:
    print("No market data found for implied volatility analysis.")

## 5. Further Analysis (e.g., Convergence, Stability)

Further analysis could involve:
- Plotting convergence of numerical methods as `n_steps` or `n_paths` increase.
- Analyzing stability of finite difference methods under different parameters.
- Comparing Greeks across different methods.
- Backtesting models against historical market prices.